### TRAIN
Entrenaremos los modelos para las n,s y m versiones de YOLO. Luego, guardar las gráficas  sobre el proceso de
entrenamiento con las que analizar su comportamiento y así establecer hipótesis como si
se produce sobreajuste o subajuste.

In [1]:
# Importar librerias necesarias
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import glob

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Fernando Pinilla\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### Configuración

In [2]:
# Ruta absoluta al dataset creado en el paso anterior
# YOLO necesita saber donde esta la carpeta 'train' y 'test'
dataset_path = os.path.abspath("dataset_isic")

# Modelos solicitados en el enunciado: nano, small, medium
modelos = ["yolov8n-cls.pt", "yolov8s-cls.pt", "yolov8m-cls.pt"]

# Hiperparametros de entrenamiento
EPOCHS = 15        # Numero de epocas de entrenamiento
IMG_SIZE = 224     # Tamaño de las imagenes (224x224 es estandar para clasificacion)

# Directorio donde se guardaran los resultados
OUTPUT_DIR = "runs/classify"

### Entrenamiento de los modelos

In [3]:
# Diccionario para almacenar los resultados de cada modelo
resultados_entrenamiento = {}

for modelo_nombre in modelos:
    print(f"\n{'='*60}")
    print(f"Iniciando entrenamiento para: {modelo_nombre}")
    print(f"{'='*60}")

    # Cargar el modelo pre-entrenado de clasificacion
    model = YOLO(modelo_nombre)

    # Entrenar el modelo
    # - data: ruta a la carpeta que contiene 'train' y 'test'
    # - epochs: numero de pasadas completas por el dataset
    # - imgsz: tamanio al que se redimensionan las imagenes
    # - project: carpeta base para guardar resultados
    # - name: nombre del experimento (subcarpeta)
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        project=OUTPUT_DIR,
        name=f"train_{modelo_nombre.replace('.pt', '')}"
    )

    # Guardar referencia a los resultados
    resultados_entrenamiento[modelo_nombre] = results
    
    print(f"\nEntrenamiento de {modelo_nombre} finalizado.")
    print(f"Resultados guardados en: {results.save_dir}")


Iniciando entrenamiento para: yolov8n-cls.pt
Ultralytics 8.3.233  Python-3.13.5 torch-2.9.1+cpu CPU (11th Gen Intel Core i9-11900K @ 3.50GHz)
Ultralytics 8.3.233  Python-3.13.5 torch-2.9.1+cpu CPU (11th Gen Intel Core i9-11900K @ 3.50GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\Fernando Pinilla\Desktop\Ingeniero\3Curso\Aprendizaje\Practicas-Aprendizaje-Automatico\src\Practica4\dataset_isic, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_rat

### Visualizacion de las graficas de entrenamiento
YOLO genera automaticamente graficas durante el entrenamiento. Vamos a cargarlas y mostrarlas para analizar el comportamiento de cada modelo.

In [9]:
def mostrar_graficas_entrenamiento(output_dir, modelos):
    """
    Muestra las graficas de entrenamiento generadas por YOLO para cada modelo.
    YOLO genera automaticamente:
    - results.png: graficas de loss y metricas por epoca
    - confusion_matrix.png: matriz de confusion del conjunto de validacion
    """
    nombres_modelos = [m.replace('.pt', '') for m in modelos]
    
    for nombre in nombres_modelos:
        # Buscar la carpeta del experimento
        carpeta_exp = os.path.join(output_dir, f"train_{nombre}")
        
        if not os.path.exists(carpeta_exp):
            print(f"No se encontro la carpeta: {carpeta_exp}")
            continue
            
        print(f"\n{'='*60}")
        print(f"Graficas para: {nombre}")
        print(f"{'='*60}")
        
        # Mostrar results.png (curvas de loss y metricas)
        results_img = os.path.join(carpeta_exp, "results.png")
        if os.path.exists(results_img):
            fig, ax = plt.subplots(figsize=(15, 10))
            img = Image.open(results_img)
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(f"Curvas de entrenamiento - {nombre}", fontsize=14)
            plt.tight_layout()
            plt.show()
        else:
            print(f"No se encontro results.png en {carpeta_exp}")
        
        # Mostrar matriz de confusion
        confusion_img = os.path.join(carpeta_exp, "confusion_matrix.png")
        if os.path.exists(confusion_img):
            fig, ax = plt.subplots(figsize=(10, 10))
            img = Image.open(confusion_img)
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(f"Matriz de Confusion - {nombre}", fontsize=14)
            plt.tight_layout()
            plt.show()

# Mostrar las graficas de todos los modelos
mostrar_graficas_entrenamiento(OUTPUT_DIR, modelos)


Graficas para: yolov8n-cls


<Figure size 1500x1000 with 1 Axes>

<Figure size 1000x1000 with 1 Axes>


Graficas para: yolov8s-cls


<Figure size 1500x1000 with 1 Axes>

<Figure size 1000x1000 with 1 Axes>


Graficas para: yolov8m-cls


<Figure size 1500x1000 with 1 Axes>

<Figure size 1000x1000 with 1 Axes>

### Analisis de las metricas de entrenamiento
Cargamos los archivos CSV con las metricas de cada epoca para comparar los modelos.

In [10]:
def cargar_metricas_entrenamiento(output_dir, modelos):
    """
    Carga los archivos results.csv generados por YOLO durante el entrenamiento.
    Estos archivos contienen las metricas de cada epoca.
    """
    nombres_modelos = [m.replace('.pt', '') for m in modelos]
    metricas = {}
    
    for nombre in nombres_modelos:
        carpeta_exp = os.path.join(output_dir, f"train_{nombre}")
        csv_path = os.path.join(carpeta_exp, "results.csv")
        
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            # Limpiar nombres de columnas (YOLO a veces agrega espacios)
            df.columns = df.columns.str.strip()
            metricas[nombre] = df
            print(f"Cargadas metricas de {nombre}: {len(df)} epocas")
        else:
            print(f"No se encontro results.csv para {nombre}")
    
    return metricas

# Cargar las metricas de todos los modelos
metricas_modelos = cargar_metricas_entrenamiento(OUTPUT_DIR, modelos)

Cargadas metricas de yolov8n-cls: 15 epocas
Cargadas metricas de yolov8s-cls: 15 epocas
Cargadas metricas de yolov8m-cls: 15 epocas


In [11]:
def comparar_modelos(metricas):
    """
    Genera graficas comparativas de las metricas entre modelos.
    Permite analizar sobreajuste/subajuste comparando train_loss vs val_loss.
    """
    if not metricas:
        print("No hay metricas disponibles para comparar")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Colores para cada modelo
    colores = {'yolov8n-cls': 'blue', 'yolov8s-cls': 'orange', 'yolov8m-cls': 'green'}
    
    for nombre, df in metricas.items():
        color = colores.get(nombre, 'gray')
        
        # Grafica 1: Loss de entrenamiento
        if 'train/loss' in df.columns:
            axes[0, 0].plot(df['epoch'], df['train/loss'], label=nombre, color=color)
        axes[0, 0].set_title('Loss de Entrenamiento por Epoca')
        axes[0, 0].set_xlabel('Epoca')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # Grafica 2: Loss de validacion
        if 'val/loss' in df.columns:
            axes[0, 1].plot(df['epoch'], df['val/loss'], label=nombre, color=color)
        axes[0, 1].set_title('Loss de Validacion por Epoca')
        axes[0, 1].set_xlabel('Epoca')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # Grafica 3: Accuracy Top-1
        if 'metrics/accuracy_top1' in df.columns:
            axes[1, 0].plot(df['epoch'], df['metrics/accuracy_top1'], label=nombre, color=color)
        axes[1, 0].set_title('Accuracy Top-1 por Epoca')
        axes[1, 0].set_xlabel('Epoca')
        axes[1, 0].set_ylabel('Accuracy')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # Grafica 4: Accuracy Top-5
        if 'metrics/accuracy_top5' in df.columns:
            axes[1, 1].plot(df['epoch'], df['metrics/accuracy_top5'], label=nombre, color=color)
        axes[1, 1].set_title('Accuracy Top-5 por Epoca')
        axes[1, 1].set_xlabel('Epoca')
        axes[1, 1].set_ylabel('Accuracy')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'comparativa_modelos.png'), dpi=150)
    plt.show()
    print(f"\nGrafica guardada en: {OUTPUT_DIR}/comparativa_modelos.png")

# Comparar los modelos
comparar_modelos(metricas_modelos)

<Figure size 1400x1000 with 4 Axes>


Grafica guardada en: runs/classify/comparativa_modelos.png


### Resumen de resultados finales
Mostramos una tabla comparativa con las metricas finales de cada modelo.

In [12]:
def mostrar_resumen_final(metricas):
    """
    Muestra una tabla con las metricas finales (ultima epoca) de cada modelo.
    """
    if not metricas:
        print("No hay metricas disponibles")
        return
    
    resumen = []
    for nombre, df in metricas.items():
        ultima_epoca = df.iloc[-1]
        resumen.append({
            'Modelo': nombre,
            'Train Loss': round(ultima_epoca.get('train/loss', 0), 4),
            'Val Loss': round(ultima_epoca.get('val/loss', 0), 4),
            'Top-1 Acc': round(ultima_epoca.get('metrics/accuracy_top1', 0), 4),
            'Top-5 Acc': round(ultima_epoca.get('metrics/accuracy_top5', 0), 4),
        })
    
    df_resumen = pd.DataFrame(resumen)
    print("\nResumen de metricas finales:")
    print("="*70)
    print(df_resumen.to_string(index=False))
    
    # Guardar como CSV
    csv_path = os.path.join(OUTPUT_DIR, 'resumen_metricas.csv')
    df_resumen.to_csv(csv_path, index=False)
    print(f"\nResumen guardado en: {csv_path}")
    
    return df_resumen

# Mostrar resumen
df_resumen = mostrar_resumen_final(metricas_modelos)


Resumen de metricas finales:
     Modelo  Train Loss  Val Loss  Top-1 Acc  Top-5 Acc
yolov8n-cls      0.8191    1.0452     0.6375     0.9625
yolov8s-cls      0.5137    1.1520     0.6500     0.9625
yolov8m-cls      0.3193    1.0287     0.6750     0.9875

Resumen guardado en: runs/classify\resumen_metricas.csv


### Analisis de sobreajuste y subajuste
Analizamos la diferencia entre train_loss y val_loss para detectar posibles problemas.

In [13]:
def analizar_sobreajuste(metricas):
    """
    Analiza si hay sobreajuste o subajuste en cada modelo.
    
    Sobreajuste (Overfitting): train_loss baja pero val_loss sube o se estanca
    Subajuste (Underfitting): tanto train_loss como val_loss son altas y no bajan
    Buen ajuste: ambas losses bajan y convergen
    """
    print("\nAnalisis de Sobreajuste/Subajuste:")
    print("="*70)
    
    for nombre, df in metricas.items():
        print(f"\n{nombre}:")
        print("-"*40)
        
        if 'train/loss' not in df.columns or 'val/loss' not in df.columns:
            print("  No hay datos suficientes para analizar")
            continue
        
        train_loss_inicio = df['train/loss'].iloc[0]
        train_loss_fin = df['train/loss'].iloc[-1]
        val_loss_inicio = df['val/loss'].iloc[0]
        val_loss_fin = df['val/loss'].iloc[-1]
        
        # Calcular la tendencia
        train_mejora = (train_loss_inicio - train_loss_fin) / train_loss_inicio * 100
        val_mejora = (val_loss_inicio - val_loss_fin) / val_loss_inicio * 100
        
        print(f"  Train Loss: {train_loss_inicio:.4f} -> {train_loss_fin:.4f} (mejora: {train_mejora:.1f}%)")
        print(f"  Val Loss:   {val_loss_inicio:.4f} -> {val_loss_fin:.4f} (mejora: {val_mejora:.1f}%)")
        
        # Detectar sobreajuste: si val_loss sube en las ultimas epocas
        val_loss_ultimas = df['val/loss'].iloc[-5:].values
        if len(val_loss_ultimas) >= 2:
            tendencia_val = val_loss_ultimas[-1] - val_loss_ultimas[0]
            if tendencia_val > 0.1:
                print("  Posible SOBREAJUSTE: val_loss aumenta en las ultimas epocas")
            elif train_loss_fin > 1.0 and val_loss_fin > 1.0:
                print("  Posible SUBAJUSTE: losses aun altas, podria necesitar mas epocas")
            else:
                print("  Buen ajuste: el modelo parece estar aprendiendo correctamente")

# Ejecutar analisis
analizar_sobreajuste(metricas_modelos)


Analisis de Sobreajuste/Subajuste:

yolov8n-cls:
----------------------------------------
  Train Loss: 2.0560 -> 0.8191 (mejora: 60.2%)
  Val Loss:   1.8868 -> 1.0452 (mejora: 44.6%)
  Buen ajuste: el modelo parece estar aprendiendo correctamente

yolov8s-cls:
----------------------------------------
  Train Loss: 2.0563 -> 0.5137 (mejora: 75.0%)
  Val Loss:   1.8264 -> 1.1520 (mejora: 36.9%)
  Buen ajuste: el modelo parece estar aprendiendo correctamente

yolov8m-cls:
----------------------------------------
  Train Loss: 2.0582 -> 0.3193 (mejora: 84.5%)
  Val Loss:   1.8867 -> 1.0287 (mejora: 45.5%)
  Buen ajuste: el modelo parece estar aprendiendo correctamente
